In [9]:
import os
import torch
import mlflow
import dagshub
from ultralytics import YOLO
from ultralytics import settings

In [ ]:
torch.cuda.empty_cache()
if mlflow.active_run():
    mlflow.end_run()

In [11]:
MY_USERNAME = "TQanh23" 
REPO_NAME = "ppe_v2"

In [12]:
dagshub.init(repo_owner=MY_USERNAME, repo_name=REPO_NAME, mlflow=True)
settings.update({'mlflow': False})
mlflow.set_experiment("PPE__yolov8m_Baseline_Evaluation")

Initialized MLflow to track repo "TQanh23/ppe_v2"

Repository TQanh23/ppe_v2 initialized!

<Experiment: artifact_location='mlflow-artifacts:/5ca5f51ddfd64c5d93284ad770344737', creation_time=1784964995164, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1784964995164, lifecycle_stage='active', name='PPE__yolov8m_Baseline_Evaluation', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [14]:
with mlflow.start_run(run_name="YOLOv8m_150e_Baseline_Val") as run:
    model_path = 'downloads/best.pt'
    model = YOLO(model_path)
    model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
    try:
        _, params, _, flops = model.info(verbose=False)
    except Exception:
        params = sum(p.numel() for p in model.model.parameters())
        flops = 0.0
    print("Đang tiến hành đánh giá và log dữ liệu lên DagsHub...")
    metrics = model.val(
        data='data.yaml',
        device=0,       
        batch=4,
        imgsz=640,
        split='val',
        project='PPE_Detection',
        name='baseline_evaluation'
    )

Exception: Run with UUID 11067f6fde81497789e74a51b2ae8ff0 is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

In [ ]:
map50 = metrics.box.map50
map50_95 = metrics.box.map
precision = metrics.box.mp
recall = metrics.box.mr
f1_score = 2 * (precision * recall) / (precision + recall + 1e-16)
inference_time = metrics.speed['inference']

In [ ]:
mlflow.log_params({
        "model_type": "YOLOv8m",
        "imgsz": 640,
        "batch_size": 4,
        "parameters": params,
        "FLOPs_G": flops,
        "model_size_MB": round(model_size_mb, 2)
    })
mlflow.log_metrics({
    "mAP_50": map50,
    "mAP_50_95": map50_95,
    "Precision": precision,
    "Recall": recall,
    "F1_score": f1_score,
    "Inference_Time_ms": inference_time
})
print("Đang upload file trọng số và đồ thị lên DagsHub...")
mlflow.log_artifact(model_path, artifact_path="models")
mlflow.log_artifacts(metrics.save_dir, artifact_path="validation_graphs")

In [ ]:
print("\n" + "="*50)
print(" BÁO CÁO KẾT QUẢ BASELINE TỔNG HỢP ".center(50))
print("="*50)
print(f"Run ID:                  {run.info.run_id}")
print("-" * 50)
print(f"mAP@50:                  {map50:.4f}")
print(f"mAP@50-95:               {map50_95:.4f}")
print(f"Precision (P):           {precision:.4f}")
print(f"Recall (R):              {recall:.4f}")
print(f"F1-score:                {f1_score:.4f}")
print("-" * 50)
print(f"Kích thước mô hình:      {model_size_mb:.2f} MB")
print(f"Số lượng tham số:        {params:,.0f}")
print(f"FLOPs:                   {flops:.1f} GFLOPs")
print(f"Thời gian suy luận:      {inference_time:.2f} ms/ảnh")
print("="*50)